In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn jupyter streamlit

In [2]:
# Importar principales librerias para el proyecto a realizar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
import sklearn
import streamlit as st
import holidays

In [3]:
# Cargar datos de inferencia
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

# Mostrar informacion del dataframe
print("Datos de inferencia:")
print(inferencia_df.head())
print(f"\nShape: {inferencia_df.shape}")
print(f"\nColumnas: {inferencia_df.columns.tolist()}")

Datos de inferencia:
        fecha producto_id                            nombre categoria  \
0  2025-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1  2025-10-25    PROD_002              Adidas Ultraboost 23   Running   
2  2025-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3  2025-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4  2025-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True               26.0   
1  Zapatillas Running          135         True               27.0   
2  Zapatillas Running           85        False                5.0   
3  Zapatillas Running           75        False                3.0   
4        Ropa Running           35        False                3.0   

   precio_venta  ingresos  Amazon  Decathlon  Deporvillage  
0        113.13   2941.38   89.51     113.43        104.78

In [4]:
# --- Transformaciones: replicar pipeline de entrenamiento ---
# Convertir fecha a datetime y crear año
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df['año'] = inferencia_df['fecha'].dt.year

# Variables temporales y festivos (misma lógica que en Entrenamineto.ipynb)
from datetime import datetime, timedelta
import holidays

do_holidays = holidays.DO()

def get_black_friday(year):
    nov = datetime(year, 11, 1)
    dec = datetime(year, 12, 1)
    last_nov = dec - timedelta(days=1)
    while last_nov.weekday() != 4:
        last_nov -= timedelta(days=1)
    return last_nov.date()

def get_cyber_monday(year):
    bf = get_black_friday(year)
    return bf + timedelta(days=3)

inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['dia_semana_num'] = inferencia_df['fecha'].dt.dayofweek
inferencia_df['fin_semana'] = (inferencia_df['dia_semana_num'] >= 5).astype(int)
inferencia_df['dia_festivo'] = inferencia_df['fecha'].dt.date.apply(lambda x: x in do_holidays).astype(int)
inferencia_df['dia_blackfriday'] = inferencia_df.apply(lambda row: row['fecha'].date() == get_black_friday(row['año']), axis=1).astype(int)
inferencia_df['dia_cyber_monday'] = inferencia_df.apply(lambda row: row['fecha'].date() == get_cyber_monday(row['año']), axis=1).astype(int)
inferencia_df['trimestre'] = ((inferencia_df['mes'] - 1) // 3) + 1
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['dia_año'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['es_bisiesto'] = inferencia_df['fecha'].dt.is_leap_year.astype(int)

# Lags y media móvil POR PRODUCTO (para que noviembre tenga histórico válido)
inferencia_df = inferencia_df.sort_values(['producto_id', 'año', 'fecha'])
for lag in range(1, 8):
    inferencia_df[f'lag_{lag}'] = inferencia_df.groupby(['producto_id', 'año'])['unidades_vendidas'].shift(lag)

inferencia_df['media_movil_7'] = inferencia_df.groupby(['producto_id', 'año'])['unidades_vendidas'].transform(lambda s: s.rolling(window=7).mean())

# Descuento, precio_competencia y ratio (misma fórmula que entrenamiento)
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']
inferencia_df.drop(columns=['Amazon', 'Decathlon', 'Deporvillage'], inplace=True)

# One hot encoding (mismo procedimiento que en entrenamiento)
inferencia_df['nombre_OHE'] = inferencia_df['nombre']
inferencia_df['categoria_OHE'] = inferencia_df['categoria']
inferencia_df['subcategoria_OHE'] = inferencia_df['subcategoria']
cols_ohe = ['nombre_OHE', 'categoria_OHE', 'subcategoria_OHE']
dummies = pd.get_dummies(inferencia_df[cols_ohe], columns=cols_ohe, prefix=cols_ohe, prefix_sep='_', dtype=int)
inferencia_df = pd.concat([inferencia_df, dummies], axis=1)

# Alinear columnas con Integration_df_Project guardado en entrenamiento
from pathlib import Path
train_path = Path('../data/Processed/Integration_df_Project.csv')
if not train_path.exists():
    train_path = Path('../data/Processed/Integration_df_Project.csv')  # fallback (misma ruta usada en entrenamiento)
train_cols = pd.read_csv(train_path, nrows=0).columns.tolist()

# Añadir columnas faltantes con 0 y eliminar columnas extras
for c in train_cols:
    if c not in inferencia_df.columns:
        inferencia_df[c] = 0
extra_cols = [c for c in inferencia_df.columns if c not in train_cols]
if extra_cols:
    inferencia_df.drop(columns=extra_cols, inplace=True)

# Reordenar columnas para que coincidan exactamente
inferencia_df = inferencia_df[train_cols]

# Mantener consistencia con la variable usada en entrenamiento
Integration_df_Project = inferencia_df.copy()

# Filtrar solo noviembre y guardar
Integration_df_Project = Integration_df_Project[Integration_df_Project['mes'] == 11].copy()

# Rellenar NaNs en lags y media móvil para permitir inferencia en noviembre (mismo esquema de columnas)
lag_cols = [f'lag_{i}' for i in range(1, 8)]
Integration_df_Project[lag_cols] = Integration_df_Project[lag_cols].fillna(0)
Integration_df_Project['media_movil_7'] = Integration_df_Project['media_movil_7'].fillna(0)
# Rellenar cualquier otro NaN numérico con 0 para mantener la estructura esperada por el modelo
Integration_df_Project = Integration_df_Project.fillna(0)

# Guardar en Data/Processed y copia lowercase
from pathlib import Path

# Guardar el dataframe Integration_df_Project procesado
dest = Path('../data/Processed')
dest.mkdir(parents=True, exist_ok=True)
ruta_salida = dest / 'df_inferencia_performence.csv'
Integration_df_Project.to_csv(ruta_salida, index=False)
# Crear también carpeta lowercase por si el usuario la esperaba

dest_lower = Path('../data/processed')
dest_lower.mkdir(parents=True, exist_ok=True)
ruta_salida_lower = dest_lower / 'df_inferencia_performence.csv'
Integration_df_Project.to_csv(ruta_salida_lower, index=False)

print(f'Procesado final guardado en: {ruta_salida} (y copia en {ruta_salida_lower})')
print('Shape final (filas x columnas):', Integration_df_Project.shape)


Procesado final guardado en: ..\data\Processed\df_inferencia_performence.csv (y copia en ..\data\processed\df_inferencia_performence.csv)
Shape final (filas x columnas): (720, 81)


In [7]:
inferencia_df.shape                       

(888, 81)

In [11]:
inferencia_df.fecha.nunique()

37

In [12]:
inferencia_df.fecha.unique()

<DatetimeArray>
['2025-10-25 00:00:00', '2025-10-26 00:00:00', '2025-10-27 00:00:00',
 '2025-10-28 00:00:00', '2025-10-29 00:00:00', '2025-10-30 00:00:00',
 '2025-10-31 00:00:00', '2025-11-01 00:00:00', '2025-11-02 00:00:00',
 '2025-11-03 00:00:00', '2025-11-04 00:00:00', '2025-11-05 00:00:00',
 '2025-11-06 00:00:00', '2025-11-07 00:00:00', '2025-11-08 00:00:00',
 '2025-11-09 00:00:00', '2025-11-10 00:00:00', '2025-11-11 00:00:00',
 '2025-11-12 00:00:00', '2025-11-13 00:00:00', '2025-11-14 00:00:00',
 '2025-11-15 00:00:00', '2025-11-16 00:00:00', '2025-11-17 00:00:00',
 '2025-11-18 00:00:00', '2025-11-19 00:00:00', '2025-11-20 00:00:00',
 '2025-11-21 00:00:00', '2025-11-22 00:00:00', '2025-11-23 00:00:00',
 '2025-11-24 00:00:00', '2025-11-25 00:00:00', '2025-11-26 00:00:00',
 '2025-11-27 00:00:00', '2025-11-28 00:00:00', '2025-11-29 00:00:00',
 '2025-11-30 00:00:00']
Length: 37, dtype: datetime64[us]

In [13]:
# --- Resumen final del dataframe procesado ---
import pandas as pd
from pathlib import Path

# Cargar el dataframe procesado
ruta_procesado = Path('../data/Processed/df_inferencia_performence.csv')
if not ruta_procesado.exists():
    ruta_procesado = Path('../data/processed/df_inferencia_performence.csv')  # fallback lowercase

df_procesado = pd.read_csv(ruta_procesado)

print("=== RESUMEN FINAL DEL DATAFRAME PROCESADO ===")
print(f"Archivo cargado: {ruta_procesado}")
print(f"Shape: {df_procesado.shape} (filas x columnas)")
print()

# Información general
print("1. Información general del DataFrame:")
print(df_procesado.info())
print()

# Estadísticas descriptivas para columnas numéricas
print("2. Estadísticas descriptivas (columnas numéricas):")
print(df_procesado.describe())
print()

# Conteo de valores únicos por columna
print("3. Valores únicos por columna:")
for col in df_procesado.columns:
    unique_count = df_procesado[col].nunique()
    print(f"{col}: {unique_count} valores únicos")
print()

# Conteo de valores nulos
print("4. Valores nulos por columna:")
null_counts = df_procesado.isnull().sum()
print(null_counts[null_counts > 0])  # Solo mostrar columnas con nulos
if null_counts.sum() == 0:
    print("No hay valores nulos en el dataframe.")
print()

# Distribución por mes y año
print("5. Distribución por mes y año:")
if 'mes' in df_procesado.columns and 'año' in df_procesado.columns:
    print(df_procesado.groupby(['año', 'mes']).size().reset_index(name='conteo'))
print()

# Conteo por categoría y subcategoría
print("6. Conteo por categoría:")
if 'categoria' in df_procesado.columns:
    print(df_procesado['categoria'].value_counts())
print()

print("7. Conteo por subcategoría:")
if 'subcategoria' in df_procesado.columns:
    print(df_procesado['subcategoria'].value_counts())
print()

# Conteo de productos estrella
print("8. Conteo de productos estrella:")
if 'es_estrella' in df_procesado.columns:
    print(df_procesado['es_estrella'].value_counts())
print()

# Rango de fechas
print("9. Rango de fechas:")
if 'fecha' in df_procesado.columns:
    fechas = pd.to_datetime(df_procesado['fecha'])
    print(f"Fecha mínima: {fechas.min()}")
    print(f"Fecha máxima: {fechas.max()}")
    print(f"Número de fechas únicas: {fechas.nunique()}")
print()

print("=== FIN DEL RESUMEN ===")

=== RESUMEN FINAL DEL DATAFRAME PROCESADO ===
Archivo cargado: ..\data\Processed\df_inferencia_performence.csv
Shape: (720, 81) (filas x columnas)

1. Información general del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 81 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   fecha                                        720 non-null    str    
 1   producto_id                                  720 non-null    str    
 2   nombre                                       720 non-null    str    
 3   categoria                                    720 non-null    str    
 4   subcategoria                                 720 non-null    str    
 5   precio_base                                  720 non-null    int64  
 6   es_estrella                                  720 non-null    bool   
 7   unidades_vendidas                            720 

In [15]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'año', 'dia_semana', 'mes', 'dia_mes', 'dia_semana_num',
       'fin_semana', 'dia_festivo', 'dia_blackfriday', 'dia_cyber_monday',
       'trimestre', 'semana_año', 'dia_año', 'es_bisiesto', 'lag_1', 'lag_2',
       'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'media_movil_7',
       'descuento_porcentaje', 'precio_competencia', 'ratio_precio',
       'nombre_OHE', 'categoria_OHE', 'subcategoria_OHE',
       'nombre_OHE_Adidas Own The Run Jacket',
       'nombre_OHE_Adidas Ultraboost 23', 'nombre_OHE_Asics Gel Nimbus 25',
       'nombre_OHE_Bowflex SelectTech 552', 'nombre_OHE_Columbia Silver Ridge',
       'nombre_OHE_Decathlon Bandas Elásticas Set', 'nombre_OHE_Domyos BM900',
       'nombre_OHE_Domyos Kit Mancuernas 20kg',
       'nombre_OHE_Gaiam Premium Yoga Block', 'nombre_OHE_Liforme Yoga Pad',
       'nombre_OHE_Lotus